# MazimaCheck: is this face forged? (OpenForensics, face crops)

**Before you run anything: File > Save a copy in Drive.** Work in your copy, not this one.
Then Runtime > Change runtime type > **T4 GPU**.

**What this notebook does.** Our first model asks "is the whole picture AI-generated?". This one asks a different question about *each face*: "is this face real, or swapped or generated?". Think of it as a second inspector who only looks at faces, with a magnifying glass.

1. Download the validation part of OpenForensics (about 3 GB) and read its labels.
2. Cut out each face the same way the app does.
3. Check for shortcuts (face size, file size) before training, the same lesson as the first project.
4. Split the photos so that near-copies of one photo never sit on both sides of the exam (version 2 fix, see section 4).
5. Train ResNet-18 and ResNet-50 and compare them.
6. Test on faces the model never saw, after chat-app style damage, and with the app's own face finder.
7. Export an ONNX file the web app can run.

The dataset is used for academic purposes only, which fits coursework. Cite: Le, Nguyen, Yamagishi, Echizen, ICCV 2021.

In [ ]:
import os, io, json, random, zipfile, subprocess, time, math, gc
import psutil
import numpy as np, pandas as pd
import torch, torch.nn as nn
from PIL import Image
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix
from sklearn.linear_model import LogisticRegression

DRY = os.environ.get("MZ_DRY") == "1"          # DRY is only for testing the code on fake data
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", dev, torch.cuda.get_device_name(0) if dev == "cuda" else "(no GPU: Runtime > Change runtime type > T4 GPU)")

ARCHS = ["resnet18"] if DRY else ["resnet18", "resnet50"]
EPOCHS = 1 if DRY else 6
BATCH = 16 if DRY else 64
SIZE, PAD, MINSIDE = 224, 0.20, 32               # PAD and SIZE must match the app (faceCrop in app.js)
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive"); OUT = "/content/drive/MyDrive/mazimacheck_faces"; WORK = "/content/of"
except Exception:
    OUT = os.environ.get("MZ_OUT", "./mazimacheck_faces"); WORK = os.environ.get("MZ_WORK", "./of")
os.makedirs(OUT, exist_ok=True); os.makedirs(WORK, exist_ok=True)
print("results are saved in:", OUT)

## 1. Get the data
This downloads straight from Zenodo into Colab's temporary disk (not your Drive), so it takes a few minutes and costs you no storage. If it stops, run the cell again: it skips what it already has.

In [ ]:
import requests
BASE = "https://zenodo.org/records/5528418/files/"
def fetch(name):
    # shows a progress bar, and if it stops you can simply run this cell again: it carries on from where it got to
    dst, url = f"{WORK}/{name}", BASE + name + "?download=1"
    total = int(requests.head(url, allow_redirects=True).headers.get("Content-Length", 0))
    have = os.path.getsize(dst) if os.path.exists(dst) else 0
    if total and have == total: print(name, "already complete"); return
    if total and have > total: os.remove(dst); have = 0
    with requests.get(url, stream=True, headers={"Range": f"bytes={have}-"} if have else {}) as r:
        if r.status_code == 416: print(name, "already complete"); return        # we already have all of it
        r.raise_for_status()
        if have and r.status_code != 206: have = 0
        with open(dst, "ab" if have else "wb") as f, tqdm(total=total or None, initial=have, unit="B", unit_scale=True, desc=name) as bar:
            for ch in r.iter_content(1 << 20): f.write(ch); bar.update(len(ch))
    if total: assert os.path.getsize(dst) == total, "incomplete download: run this cell again"
if not DRY:
    fetch("Val_poly.json"); fetch("Val.zip")
    if not os.path.isdir(f"{WORK}/Val"):
        assert zipfile.is_zipfile(f"{WORK}/Val.zip"), "Val.zip is incomplete: run this cell again"
        print("unzipping (a minute or two)...")
        zipfile.ZipFile(f"{WORK}/Val.zip").extractall(f"{WORK}/Val")
print(os.listdir(WORK))

## 2. Read the labels (and do not trust our guesses)
The annotation file is in COCO format. We print its categories and map them to real (0) or forged (1) by their **names**, and stop if the names are not what we expect.

In [ ]:
coco = json.load(open(f"{WORK}/Val_poly.json"))
print("keys:", list(coco.keys()))
print("categories:", coco["categories"])
print("images:", len(coco["images"]), "annotations:", len(coco["annotations"]))
print("first image:", coco["images"][0])
print("first annotation:", {k: (v if k != "segmentation" else "...") for k, v in coco["annotations"][0].items()})

LABEL = {}
for c in coco["categories"]:
    n = c["name"].lower()
    LABEL[c["id"]] = 1 if ("fake" in n or "forg" in n) else (0 if "real" in n else None)
print("label map (0 real, 1 forged):", {c["name"]: LABEL[c["id"]] for c in coco["categories"]})
assert set(LABEL.values()) == {0, 1}, "Category names are not what we expected. Read the printout above and fix LABEL by hand."

## 3. Cut out the faces (square crops)
Version 2 change: the first run squashed every face box into a square, so a wide box looked like a stretched face, and box shape separated real from forged (AUC 0.70). Now the crop is a **square** around the face centre, with 20% extra on each side, so faces are never stretched. Parts that fall outside the photo are black, and the app does the same. Faces whose shorter side is under 32 pixels are skipped. Crops are PNG so no extra compression sneaks in.

In [ ]:
path_of = {}
for r, _, fs in os.walk(f"{WORK}/Val"):
    for f in fs: path_of[f] = os.path.join(r, f)
imgs = {i["id"]: i for i in coco["images"]}
by_img = {}
for a in coco["annotations"]:
    if LABEL.get(a["category_id"]) is None or "bbox" not in a: continue
    by_img.setdefault(a["image_id"], []).append(a)

def square_crop(im, x, y, w, h):
    cx, cy, s = x + w / 2, y + h / 2, max(w, h) * (1 + 2 * PAD)
    box = (int(round(cx - s / 2)), int(round(cy - s / 2)), int(round(cx + s / 2)), int(round(cy + s / 2)))
    return im.crop(box).resize((SIZE, SIZE), Image.BILINEAR)           # anything outside the photo is black

CACHE = f"{OUT}/cache_v2"; os.makedirs(CACHE, exist_ok=True)      # saved progress in your Drive: if Colab drops, re-running skips what is done
CROPS = f"{WORK}/crops_sq"; os.makedirs(CROPS, exist_ok=True)
df = None
if os.path.exists(f"{CACHE}/faces_df.csv") and os.path.exists(f"{CACHE}/crops_sq.tar"):
    print("restoring the cut faces saved in your Drive (about a minute)...")
    subprocess.run(["tar", "-xf", f"{CACHE}/crops_sq.tar", "-C", WORK], check=True)
    df = pd.read_csv(f"{CACHE}/faces_df.csv"); df["path"] = CROPS + "/" + df["name"]
    if not all(os.path.exists(p) for p in df.path.sample(min(50, len(df)), random_state=0)): df = None; print("saved faces were incomplete, cutting again")
if df is None:
    rows, skipped = [], 0
    for iid, anns in tqdm(by_img.items()):
        meta = imgs[iid]; p = path_of.get(os.path.basename(meta["file_name"]))
        if p is None: skipped += len(anns); continue
        im = Image.open(p).convert("RGB"); W, H = im.size; src_bytes = os.path.getsize(p)
        for k, a in enumerate(anns):
            x, y, w, h = a["bbox"]
            if min(w, h) < MINSIDE: skipped += 1; continue
            name = f"{iid}_{k}.png"
            square_crop(im, x, y, w, h).save(f"{CROPS}/{name}")
            rows.append(dict(name=name, path=f"{CROPS}/{name}", image_id=iid, label=LABEL[a["category_id"]], x=x, y=y, w=w, h=h, srcW=W, srcH=H, src_bpp=src_bytes / (W * H)))
    df = pd.DataFrame(rows); print(f"{skipped} faces skipped (too small or image missing)")
    subprocess.run(["tar", "-cf", f"{CACHE}/crops_sq.tar", "-C", WORK, "crops_sq"], check=True)   # tar first, csv last: the csv means 'finished'
    df.drop(columns=["path"]).to_csv(f"{CACHE}/faces_df.csv", index=False)
    print("saved the cut faces to your Drive")
del by_img; coco["annotations"] = None; gc.collect()                       # free memory: the big label file is no longer needed
print(f"{len(df)} face crops")
print(df.label.value_counts().rename({0: "real", 1: "forged"}))

## 4. Fair exam: group the photos, then split, then audit for shortcuts
**Why this changed.** In the first run, 50% of the test photos had a near-identical twin in the training photos (OpenForensics makes its forged images by editing real photos, so one background appears many times). That makes the exam too easy, like practising on the exact exam questions. So we now:
1. give every photo a tiny fingerprint (a 16 x 16 'difference hash'),
2. put photos whose fingerprints are nearly the same into one **group**,
3. split by group, so a photo and its edited twins always stay on the same side.

Then the shortcut audit: **how well can face size, box shape or photo size alone separate real from forged?** Anything far above 0.5 is a shortcut, and we balance the training set against it.

In [ ]:
def dhash(path):
    im = Image.open(path); im.draft("L", (80, 80)); im = im.convert("L").resize((17, 16), Image.BILINEAR)
    a = np.asarray(im, dtype=np.int16); return (a[:, 1:] > a[:, :-1]).flatten()
photo = df.drop_duplicates("image_id").set_index("image_id")
pids = list(photo.index)
hf = f"{CACHE}/hashes.npz"
if os.path.exists(hf) and np.load(hf)["pids"].tolist() == [int(p) for p in pids]: Hm = np.load(hf)["H"]
else:
    Hm = np.stack([dhash(path_of[os.path.basename(imgs[i]["file_name"])]) for i in tqdm(pids)]).astype(np.float32); np.savez(hf, H=Hm, pids=np.array([int(p) for p in pids]))
NB = Hm.shape[1]; THR = 30                                   # photos closer than 30 of 256 bits are treated as the same photo
def bit_distance(A, B): return NB - (A @ B.T + (1 - A) @ (1 - B).T)

parent = list(range(len(pids)))
def find(a):
    while parent[a] != a: parent[a] = parent[parent[a]]; a = parent[a]
    return a
for k in range(0, len(pids), 500):
    ii, jj = np.nonzero(bit_distance(Hm[k:k + 500], Hm) <= THR)
    for i, j in zip(ii + k, jj):
        if i < j: parent[find(i)] = find(j)
members = {}
for idx, pid in enumerate(pids): members.setdefault(find(idx), []).append(pid)
groups = list(members.values()); random.Random(SEED).shuffle(groups); groups.sort(key=len, reverse=True)
print(f"{len(pids)} photos fall into {len(groups)} groups; the biggest group has {len(groups[0])} photos; {sum(len(g) for g in groups if len(g) > 1)} photos have at least one twin")

targets = {"train": .70, "val": .15, "test": .15}; counts = {k: 0 for k in targets}; split_of = {}
for g in groups:
    k = min(targets, key=lambda s: counts[s] / targets[s]); counts[k] += len(g)
    for pid in g: split_of[pid] = k
df["split"] = df.image_id.map(split_of)
print(df.groupby(["split", "label"]).size().unstack())
pos = {pid: n for n, pid in enumerate(pids)}
tr_idx = [pos[p] for p in pids if split_of[p] == "train"]; te_idx = [pos[p] for p in pids if split_of[p] == "test"]
closest = np.concatenate([bit_distance(Hm[te_idx][k:k + 200], Hm[tr_idx]).min(1) for k in range(0, len(te_idx), 200)])
near = int((closest <= THR).sum())
print(f"proof: {near} of {len(te_idx)} test photos have a near-twin in the training photos (before the fix it was 50%); closest test-to-train distance is now {int(closest.min())} bits")

In [ ]:
df["logarea"] = np.log(df.w * df.h); df["aspect"] = df.w / df.h
def feats(d, cols): return d[cols].values
def auc_of(col, d):
    a = roc_auc_score(d.label, d[col]); return max(a, 1 - a)          # which way round does not matter
tr0, te0 = df[df.split == "train"], df[df.split == "test"]
print("single feature AUC on the test faces (0.5 = no signal)")
for col in ["logarea", "aspect", "srcW", "srcH", "src_bpp"]: print(f"  {col:8s} {auc_of(col, te0):.3f}")
print("face box shape (width / height) by class:"); print(df.groupby("label").aspect.describe()[["25%", "50%", "75%"]].rename(index={0: "real", 1: "forged"}).round(2))

In [ ]:
def balance(d, seed=SEED):
    # inside each (face size x box shape) cell, keep equal numbers of real and forged, so neither says anything about the label
    def qbin(x): return pd.qcut(x, 4, labels=False, duplicates="drop").fillna(0) if x.nunique() > 4 else x * 0       # 4 equal-sized bins
    d = d.copy(); d["a"], d["b"] = qbin(d.logarea), qbin(d.aspect); out = []
    for _, g in d.groupby(["a", "b"]):
        k = int(g.label.value_counts().reindex([0, 1], fill_value=0).min())
        if k == 0: continue                                   # a cell with only one class cannot be balanced
        out += [g[g.label == c].sample(k, random_state=seed) for c in (0, 1)]
    return pd.concat(out).drop(columns=["a", "b"])
train_df = balance(df[df.split == "train"])
val_df, test_df = df[df.split == "val"], df[df.split == "test"]
test_bal = balance(test_df)
print("train after balancing:", train_df.label.value_counts().to_dict(), "| test:", len(test_df), "| balanced test:", len(test_bal))
tc = ["logarea", "aspect"]; mu, sd = feats(train_df, tc).mean(0), feats(train_df, tc).std(0) + 1e-9
clf = LogisticRegression(max_iter=1000).fit((feats(train_df, tc) - mu) / sd, train_df.label)
print("face size + box shape AUC on the balanced test set:", round(roc_auc_score(test_bal.label, clf.predict_proba((feats(test_bal, tc) - mu) / sd)[:, 1]), 3), "(should be near 0.5)")

Here is a quick look at what the model will see. Top row real, bottom row forged. Do the forged ones look obviously fake to *you*? Write that down for the report.

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(2, 8, figsize=(16, 4.4))
for r, lab in enumerate((0, 1)):
    s = train_df[train_df.label == lab].sample(min(8, (train_df.label == lab).sum()), random_state=1)
    for c, p in enumerate(s.path):
        ax[r, c].imshow(Image.open(p)); ax[r, c].axis("off")
    ax[r, 0].set_title("real" if lab == 0 else "forged", loc="left")
plt.tight_layout(); plt.show()

## 5. Train
We train ResNet-18 and ResNet-50 the same way, so the comparison is fair. Training pictures are randomly flipped, colour-shifted, shrunk and re-compressed, so the model has seen chat-app damage before the exam. The best epoch is chosen on the **validation** faces, never on the test faces.

In [ ]:
import torchvision as tv
from torch.utils.data import Dataset, DataLoader

def damage(im, shrink=None, quality=None):
    if shrink: im = im.resize((shrink, shrink), Image.BILINEAR).resize((SIZE, SIZE), Image.BILINEAR)
    if quality:
        b = io.BytesIO(); im.save(b, "JPEG", quality=quality); b.seek(0); im = Image.open(b).convert("RGB")
    return im
norm = tv.transforms.Compose([tv.transforms.ToTensor(), tv.transforms.Normalize(MEAN, STD)])
jitter = tv.transforms.ColorJitter(.2, .2, .2)

class Faces(Dataset):
    def __init__(self, d, train=False, fixed=None): self.p, self.y, self.train, self.fixed = list(d.path), list(d.label), train, fixed
    def __len__(self): return len(self.p)
    def __getitem__(self, i):
        im = Image.open(self.p[i]).convert("RGB")
        if self.train:
            if random.random() < .5: im = im.transpose(Image.FLIP_LEFT_RIGHT)
            im = jitter(im)
            im = damage(im, random.randint(96, SIZE) if random.random() < .5 else None, random.randint(40, 95) if random.random() < .5 else None)
        elif self.fixed: im = damage(im, *self.fixed)
        return norm(im), torch.tensor(float(self.y[i]))
def loader(d, train=False, fixed=None):
    return DataLoader(Faces(d, train, fixed), batch_size=BATCH, shuffle=train, num_workers=0 if DRY else 2, drop_last=False)

def make(arch):
    m = getattr(tv.models, arch)(weights=None if DRY else "DEFAULT"); m.fc = nn.Linear(m.fc.in_features, 1); return m.to(dev)

@torch.no_grad()
def predict(m, dl):
    m.eval(); P = []
    for x, _ in dl: P.append(torch.sigmoid(m(x.to(dev)).squeeze(1)).float().cpu())
    return torch.cat(P).numpy()
def best_threshold(p, y):
    ts = np.linspace(.05, .95, 91); f = [f1_score(y, p >= t, average="macro") for t in ts]; return float(ts[int(np.argmax(f))]), float(max(f))

results, models = {}, {}
for arch in ARCHS:
    ck, cj = f"{CACHE}/{arch}.pt", f"{CACHE}/{arch}.json"
    if os.path.exists(ck) and os.path.exists(cj):
        m = make(arch); m.load_state_dict(torch.load(ck, map_location=dev)); models[arch] = m; results[arch] = json.load(open(cj))
        print(arch, "was already trained: loaded from your Drive"); continue
    m = make(arch); opt = torch.optim.AdamW(m.parameters(), lr=2e-4, weight_decay=1e-4)
    dl = loader(train_df, train=True); sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-4, total_steps=EPOCHS * len(dl))
    scaler = torch.amp.GradScaler(enabled=(dev == "cuda")); lossf = nn.BCEWithLogitsLoss(); best = (-1, None, None)
    t0 = time.time()
    for ep in range(EPOCHS):
        m.train(); tot = 0
        for x, y in tqdm(dl, desc=f"{arch} epoch {ep + 1}/{EPOCHS}", leave=False):
            x, y = x.to(dev), y.to(dev)
            with torch.autocast(device_type=dev, enabled=(dev == "cuda")): loss = lossf(m(x).squeeze(1), y)
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step(); tot += loss.item() * len(x)
        pv = predict(m, loader(val_df)); th, f1 = best_threshold(pv, val_df.label.values)
        print(f"{arch} epoch {ep + 1}: train loss {tot / len(train_df):.3f} | val macro-F1 {f1:.3f} at threshold {th:.2f} | val AUC {roc_auc_score(val_df.label, pv):.3f} | {time.time() - t0:.0f}s | memory in use {psutil.virtual_memory().percent:.0f}%")
        if f1 > best[0]: best = (f1, th, {k: v.detach().cpu().clone() for k, v in m.state_dict().items()})
    m.load_state_dict(best[2]); models[arch] = m; results[arch] = dict(val_f1=best[0], threshold=best[1])
    torch.save(best[2], ck); json.dump(results[arch], open(cj, "w"))      # saved: a dropped connection no longer loses this model

## 6. The exam: faces the model has never seen
Same test faces for both models. We also re-test after damage (a small, heavily compressed copy, like a face from a forwarded WhatsApp video) and on the **size-balanced** test set, where face size cannot help.

In [ ]:
def report(m, d, th, fixed=None):
    p = predict(m, loader(d, fixed=fixed)); y = d.label.values
    return dict(f1=f1_score(y, p >= th, average="macro"), auc=roc_auc_score(y, p), cm=confusion_matrix(y, p >= th), p=p)
table = []
for arch, m in models.items():
    th = results[arch]["threshold"]
    for name, d, fx in [("test, as cut", test_df, None), ("test, size-balanced", test_bal, None),
                        ("test, damaged (112 px, JPEG q45)", test_df, (112, 45)), ("test, JPEG q75 only", test_df, (None, 75))]:
        r = report(m, d, th, fx); results[arch][name] = {k: v for k, v in r.items() if k != "p"}
        table.append(dict(model=arch, test=name, macroF1=round(r["f1"], 3), AUC=round(r["auc"], 3)))
        if name == "test, as cut": results[arch]["test_p"] = r["p"]
print(pd.DataFrame(table).to_string(index=False))
for arch in models:
    print(f"\n{arch} confusion matrix on the test faces (rows real/forged, columns predicted real/forged), threshold {results[arch]['threshold']:.2f}")
    print(results[arch]["test, as cut"]["cm"])
pick = max(results, key=lambda a: results[a]["val_f1"]); print("\nchosen by validation score:", pick, round(results[pick]["val_f1"], 4))

## 7. The real-life test: the app's own face finder
In the app, faces are not found by the dataset's hand-drawn boxes but by a small detector (YuNet). So here we run the **same detector** on test photos, cut the crops the same way the app does, and score them. This shows what happens when the app meets a photo, not just when the notebook meets a tidy crop. We compare the scores from the dataset's boxes and from the detector's boxes for the same faces.

In [ ]:
import cv2
YU = f"{WORK}/yunet.onnx"
if not os.path.exists(YU) or os.path.getsize(YU) < 100000:
    subprocess.run(["wget", "-q", "-O", YU, "https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx"], check=True)
detector = cv2.FaceDetectorYN.create(YU, "", (640, 640), 0.7, 0.3, 5000)       # same size and thresholds as the app
def letterbox(bgr):
    h, w = bgr.shape[:2]; sc = min(640 / w, 640 / h); nw, nh = round(w * sc), round(h * sc)
    c = np.zeros((640, 640, 3), np.uint8); c[:nh, :nw] = cv2.resize(bgr, (nw, nh), interpolation=cv2.INTER_AREA if sc < 1 else cv2.INTER_LINEAR); return c, sc
def box_iou(a, b):
    x1, y1, x2, y2 = max(a[0], b[0]), max(a[1], b[1]), min(a[0] + a[2], b[0] + b[2]), min(a[1] + a[3], b[1] + b[3])
    i = max(0, x2 - x1) * max(0, y2 - y1); return i / (a[2] * a[3] + b[2] * b[3] - i + 1e-9)
@torch.no_grad()
def score_crops(m, crops):
    m.eval(); out = []
    for k in range(0, len(crops), 64): out.append(torch.sigmoid(m(torch.stack([norm(c) for c in crops[k:k + 64]]).to(dev)).squeeze(1)).float().cpu())
    return torch.cat(out).numpy() if out else np.array([])

tids = sorted(test_df.image_id.unique()); random.Random(SEED).shuffle(tids); tids = tids[: (6 if DRY else 500)]
gt_total, kept, c_gt, c_det, lab = 0, 0, [], [], []
for iid in tqdm(tids):
    p = path_of[os.path.basename(imgs[iid]["file_name"])]; bgr = cv2.imread(p); lb, sc = letterbox(bgr); _, det = detector.detect(lb)
    gts = test_df[test_df.image_id == iid]; gt_total += len(gts)
    if det is None: continue
    dets = [tuple(d[:4] / sc) for d in det]; im = Image.open(p).convert("RGB")
    for g in gts.itertuples():
        best = max(dets, key=lambda d: box_iou(d, (g.x, g.y, g.w, g.h)))
        if box_iou(best, (g.x, g.y, g.w, g.h)) < 0.4: continue
        c_gt.append(square_crop(im, g.x, g.y, g.w, g.h)); c_det.append(square_crop(im, *best)); lab.append(g.label); kept += 1
print(f"the face finder located {kept} of {gt_total} labelled faces in {len(tids)} test photos ({100 * kept / max(1, gt_total):.0f}%); the rest are small, turned away or missed")
e2e = {"found_share": kept / max(1, gt_total), "faces_matched": kept}
if kept >= 20 and len(set(lab)) == 2:
    th = results[pick]["threshold"]; m = models[pick]; lab = np.array(lab)
    for nm, cr in (("dataset boxes", c_gt), ("app face finder boxes", c_det)):
        p = score_crops(m, cr); e2e[nm] = dict(f1=float(f1_score(lab, p >= th, average="macro")), auc=float(roc_auc_score(lab, p)))
        print(f"  {nm:24s} macro-F1 {e2e[nm]['f1']:.3f}   AUC {e2e[nm]['auc']:.3f}   (same {kept} faces)")
else: print("not enough matched faces of both kinds for a fair comparison")

## 8. Set the verdict zones and export for the app
We pick the model with the better **validation** score. If ResNet-18 is close to ResNet-50, prefer ResNet-18: it is about half the download. The zones work like the photo model: a score below `real_below` reads "likely real", above `ai_from` reads "likely forged", and the middle is "uncertain". They are chosen on validation faces so that each side is right about 80% of the time, when that is possible.

In [ ]:
th = results[pick]["threshold"]
pv = predict(models[pick].to(dev), loader(val_df)); yv = val_df.label.values
def zones(p, y, target=.80):
    hi = [t for t in np.linspace(.5, .99, 99) if (p >= t).sum() >= .05 * len(p) and y[p >= t].mean() >= target]
    lo = [t for t in np.linspace(.01, .5, 99) if (p < t).sum() >= .05 * len(p) and (1 - y[p < t]).mean() >= target]
    return (max(lo) if lo else .45), (min(hi) if hi else .65)
real_below, forged_from = zones(pv, yv); print(f"zones: likely real below {real_below:.2f}, likely forged from {forged_from:.2f}")

class Wrap(nn.Module):
    def __init__(s, m): super().__init__(); s.m = m
    def forward(s, x): return s.m(x).squeeze(-1)
m = models[pick].cpu().eval()                                    # export on the CPU (the model was on the GPU for scoring)
onnx_path = f"{OUT}/faces_{pick}.onnx"; dummy = torch.randn(2, 3, SIZE, SIZE); w = Wrap(m).eval()
kw = dict(input_names=["input"], output_names=["logit"], dynamic_axes={"input": {0: "n"}, "logit": {0: "n"}}, opset_version=17)
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime"], check=True)      # Colab does not ship with these
try: torch.onnx.export(w, dummy, onnx_path, dynamo=False, **kw)
except TypeError: torch.onnx.export(w, dummy, onnx_path, **kw)
import onnxruntime as ort
s = ort.InferenceSession(onnx_path); x = torch.randn(4, 3, SIZE, SIZE)
diff = float(np.abs(s.run(None, {"input": x.numpy()})[0] - w(x).detach().numpy()).max())
print(f"ONNX saved: {onnx_path}  ({os.path.getsize(onnx_path) / 1e6:.1f} MB), max logit difference vs PyTorch {diff:.2e}")

summary = dict(model=pick, onnx=os.path.basename(onnx_path), crop=dict(size=SIZE, pad=PAD, min_side=MINSIDE, square=True), end_to_end_with_app_face_finder=e2e, threshold=th,
               zones=dict(real_below=real_below, forged_from=forged_from), onnx_parity_max_logit_diff=diff,
               compared={a: {k: (v if not isinstance(v, dict) else {kk: (vv.tolist() if hasattr(vv, "tolist") else vv) for kk, vv in v.items()}) for k, v in r.items() if k != "test_p"} for a, r in results.items()},
               data=dict(source="OpenForensics v1.0.0 Val split", faces=int(len(df)), train=int(len(train_df)), val=int(len(val_df)), test=int(len(test_df))))
json.dump(summary, open(f"{OUT}/faces_summary.json", "w"), indent=1, default=float)
df.drop(columns=["path"]).to_csv(f"{OUT}/faces_split.csv", index=False)
print("saved faces_summary.json and faces_split.csv in", OUT)

## 9. Optional: a second dataset the model has never met
The honest test is a different source. Put faces in your Drive like this and run the cell:

`MyDrive/mazimacheck_faces/external/real/*.jpg` and `MyDrive/mazimacheck_faces/external/fake/*.jpg`

Faces should already be cropped. Anything you find must be allowed for research use. Report the result whatever it is: a drop is a finding, not a failure.

In [ ]:
import glob
EXT = f"{OUT}/external"
files = [(p, lab) for lab, nm in ((0, "real"), (1, "fake")) for p in glob.glob(f"{EXT}/{nm}/*")]
if not files: print("No external folder yet. Skipping.")
else:
    rows = []
    for p, lab in files:
        try: im = Image.open(p).convert("RGB").resize((SIZE, SIZE), Image.BILINEAR); q = f"{WORK}/ext_{len(rows)}.png"; im.save(q); rows.append(dict(path=q, label=lab))
        except Exception as e: print("skip", p, e)
    ext = pd.DataFrame(rows); r = report(models[pick].to(dev), ext, th)
    print(f"external faces: {len(ext)}  macro-F1 {r['f1']:.3f}  AUC {r['auc']:.3f}"); print(r["cm"])